In [ ]:
import struct
from PIL import Image, ImageDraw

data = open('../COMM.IMG', 'rb').read()
print(len(data))

row_width = 896
# right_pixels = 192   
right_pixels = 130  
# right_pixels = int(right_bytes / 2)
right_bytes = right_pixels * 2
left_bytes = row_width - right_bytes    
left_pixels = int(left_bytes * 2)
print(f"left_bytes: {left_bytes}, left_pixels: {left_pixels}")
print(f"right_bytes: {right_bytes}, right_pixels: {right_pixels}")

img = Image.new('RGB', (left_pixels + right_pixels, 512), (0, 0, 0))
pix = img.load()

# 320 x 240
w = 320
# w = 192
h = 240
img2 = Image.new('RGB', (w, h), (0, 0, 0))
pix2 = img2.load()

for y in range(512): # byte
    # Left: 4bpp indexed (grayscale without runtime CLUT)
    for xb in range(left_bytes):
        offset = y * row_width + xb
        byte = data[offset]
        lo = (byte & 0x0F) * 17
        hi = ((byte >> 4) & 0x0F) * 17
        # lo = (byte & 0x0F) * 10
        # hi = ((byte >> 4) & 0x0F) * 10
        pix[xb * 2, y] = (lo, lo, lo)
        pix[xb * 2 + 1, y] = (hi, hi, hi)

    # Right: 16-bit RGB555
    for x in range(right_pixels):
        offset = y * row_width + left_bytes + x * 2
        val = struct.unpack_from('<H', data, offset)[0]
        r = (val & 0x1F) << 3
        g = ((val >> 5) & 0x1F) << 3
        b = ((val >> 10) & 0x1F) << 3
        pix[left_pixels + x, y] = (r, g, b)

x_cut = 192
left_bytes = row_width - x_cut * 2
for y in range(h):
    for x in range(w):
        if x < x_cut:
            offset = y * row_width + left_bytes + x * 2
            val = struct.unpack_from('<H', data, offset)[0]
            r = (val & 0x1F) << 3
            g = ((val >> 5) & 0x1F) << 3
            b = ((val >> 10) & 0x1F) << 3
            pix2[x, y] = (r, g, b)
        else:
            offset = (y+h+16) * row_width + left_bytes + (x-x_cut+64) * 2
            # offset = (y) * row_width + left_bytes + x * 2
            val = struct.unpack_from('<H', data, offset)[0]
            r = (val & 0x1F) << 3
            g = ((val >> 5) & 0x1F) << 3
            b = ((val >> 10) & 0x1F) << 3
            pix2[x, y] = (r, g, b)

# Draw bounding boxes for left and right regions
draw = ImageDraw.Draw(img)

# Left region box (4bpp): red
draw.rectangle([0, 0, left_pixels - 1, 511], outline='red', width=2)
draw.text((4, 4), f"4bpp ({left_pixels}x512)", fill='red')

# Right region box (16-bit): cyan
draw.rectangle([left_pixels, 0, left_pixels + right_pixels - 1, 511], outline='cyan', width=2)
draw.text((left_pixels + 4, 4), f"16-bit RGB555 ({right_pixels}x512)", fill='cyan')

img.show()
img2.show()

458752
left_bytes: 636, left_pixels: 1272
right_bytes: 260, right_pixels: 130


In [165]:
# Draw the 16-color grayscale palette used for the 4bpp left region
from PIL import ImageFont

swatch_w, swatch_h = 40, 40
label_h = 20
palette_img = Image.new('RGB', (16 * swatch_w, swatch_h + label_h), (0, 0, 0))
draw_p = ImageDraw.Draw(palette_img)

for i in range(16):
    # gray = i * 17  # same mapping as lo/hi: index * 17 -> 0,17,34,...,255
    gray = 255 - i * 17  # same mapping as lo/hi: index * 17 -> 0,17,34,...,255
    x0 = i * swatch_w
    x1 = x0 + swatch_w - 1
    draw_p.rectangle([x0, 0, x1, swatch_h - 1], fill=(gray, gray, gray))
    # draw index label below each swatch
    text_color = 'white' if gray < 128 else 'black'
    draw_p.text((x0 + 2, swatch_h + 2), f"{i}\n({gray})", fill='white')

palette_img.show()
print("4bpp grayscale palette: index * 17 -> [0, 17, 34, 51, ..., 238, 255]")


4bpp grayscale palette: index * 17 -> [0, 17, 34, 51, ..., 238, 255]


In [ ]:
# Extract top-left 256x256 font area and draw 21x21 character grid
# Characters: 11x11 px, 1px padding between them, 5px unused at right edge
# 21 chars * 11px + 20 gaps * 1px = 251px + 5px margin = 256px

char_w, char_h = 11, 11
pad = 1
grid_cols, grid_rows = 21, 21

# Extract font area from COMM.IMG (4bpp, first 128 bytes x 256 rows)
font_area = Image.new('RGB', (256, 256), (0, 0, 0))
fp = font_area.load()
for y in range(256):
    for xb in range(128):
        offset = y * row_width + xb
        byte = data[offset]
        lo = (byte & 0x0F) * 17
        hi = ((byte >> 4) & 0x0F) * 17
        fp[xb * 2, y] = (lo, lo, lo)
        fp[xb * 2 + 1, y] = (hi, hi, hi)

# Scale up for visibility
scale = 3
font_scaled = font_area.resize((256 * scale, 256 * scale), Image.NEAREST)
draw_grid = ImageDraw.Draw(font_scaled)

# Draw grid lines
for col in range(grid_cols + 1):
    x = col * (char_w + pad) * scale
    color = 'red' if col < grid_cols else 'yellow'
    draw_grid.line([(x, 0), (x, 256 * scale - 1)], fill=color, width=1)
    # right edge of each character cell
    if col < grid_cols:
        x_right = (col * (char_w + pad) + char_w) * scale
        draw_grid.line([(x_right, 0), (x_right, 256 * scale - 1)], fill='green', width=1)

for row in range(grid_rows + 1):
    y = row * (char_h + pad) * scale
    color = 'red' if row < grid_rows else 'yellow'
    draw_grid.line([(0, y), (256 * scale - 1, y)], fill=color, width=1)
    # bottom edge of each character cell
    if row < grid_rows:
        y_bottom = (row * (char_h + pad) + char_h) * scale
        draw_grid.line([(0, y_bottom), (256 * scale - 1, y_bottom)], fill='green', width=1)

# Mark the 5px margin area at right
margin_x = (grid_cols * (char_w + pad) - pad) * scale  # where last char ends + its padding
# actually: 21 * 11 + 20 * 1 = 251, so margin starts at pixel 251
margin_start = 251 * scale
draw_grid.rectangle([margin_start, 0, 256 * scale - 1, 256 * scale - 1], outline='blue', width=1)

font_scaled.show()
print(f"Grid: {grid_cols}x{grid_rows} chars, each {char_w}x{char_h}px, {pad}px padding")
print(f"Used area: {grid_cols * char_w + (grid_cols-1) * pad}x{grid_rows * char_h + (grid_rows-1) * pad} = 251x251")
print(f"Margin: 5px right, 5px bottom")


In [ ]:
import os

# Font uses BITPLANE ENCODING: 4bpp index = 4-bit mask, each bit is a separate character
# Bit 0 (& 1): Layer 0
# Bit 1 (& 2): Layer 1
# Bit 2 (& 4): Layer 2
# Bit 3 (& 8): Layer 3
# Where characters overlap, indices add up (e.g. index 15 = all 4 layers)
# Total: 441 cells x 4 bitplanes = 1764 characters

char_w, char_h = 11, 11
pad = 1
grid_cols, grid_rows = 21, 21

# Build 256x256 index map
indices = [[0]*256 for _ in range(256)]
for y in range(256):
    for xb in range(128):
        offset = y * row_width + xb
        byte = data[offset]
        indices[y][xb * 2] = byte & 0x0F
        indices[y][xb * 2 + 1] = (byte >> 4) & 0x0F

# Extract 4 bitplane layers
for bit in range(4):
    mask = 1 << bit
    out_dir = f'chars_bit{bit}'
    os.makedirs(out_dir, exist_ok=True)
    count = 0
    for grow in range(grid_rows):
        for gcol in range(grid_cols):
            x0 = gcol * (char_w + pad)
            y0 = grow * (char_h + pad)
            if grow == grid_rows - 1 and gcol == grid_cols - 1:
                w, h = 256 - x0, 256 - y0
            else:
                w, h = char_w, char_h

            img = Image.new('L', (w, h), 0)
            px = img.load()
            for dy in range(h):
                for dx in range(w):
                    idx = indices[y0 + dy][x0 + dx]
                    if idx & mask:
                        px[dx, dy] = 255
            img.save(os.path.join(out_dir, f'char_{count:03d}.png'))
            count += 1
    print(f'chars_bit{bit}/: saved {count} chars (mask={mask})')

print(f'\nTotal: {grid_rows * grid_cols * 4} characters (441 cells x 4 bitplanes)')
